In [9]:
import cv2
import mediapipe as mp
import numpy as np
import os
import csv
from math import sqrt


In [10]:
mp_face_mesh = mp.solutions.face_mesh
mp_hands = mp.solutions.hands

face_mesh = mp_face_mesh.FaceMesh(
    static_image_mode=True,
    refine_landmarks=True  
)

hands = mp_hands.Hands(static_image_mode=True, max_num_hands=1)


I0000 00:00:1769071181.535717       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M2
I0000 00:00:1769071181.547676       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M2


In [11]:
#eyes
LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]

#iris
LEFT_IRIS = [468, 469, 470, 471]
RIGHT_IRIS = [472, 473, 474, 475]

#face
NOSE_TIP = 1
LEFT_FACE = 234
RIGHT_FACE = 454


In [12]:
def euclidean(p1, p2):
    return sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

def eye_aspect_ratio(eye):
    A = euclidean(eye[1], eye[5])
    B = euclidean(eye[2], eye[4])
    C = euclidean(eye[0], eye[3])
    return (A + B) / (2.0 * C)


In [13]:
def get_head_pose(image, face_landmarks):
    h, w, _ = image.shape

    image_points = np.array([
        (face_landmarks[1].x * w, face_landmarks[1].y * h),     # Nose
        (face_landmarks[152].x * w, face_landmarks[152].y * h), # Chin
        (face_landmarks[33].x * w, face_landmarks[33].y * h),   # Left eye
        (face_landmarks[263].x * w, face_landmarks[263].y * h), # Right eye
        (face_landmarks[61].x * w, face_landmarks[61].y * h),   # Mouth left
        (face_landmarks[291].x * w, face_landmarks[291].y * h)  # Mouth right
    ], dtype="double")

    model_points = np.array([
        (0, 0, 0),
        (0, -63.6, -12.5),
        (-43.3, 32.7, -26.0),
        (43.3, 32.7, -26.0),
        (-28.9, -28.9, -24.1),
        (28.9, -28.9, -24.1)
    ])

    focal_length = w
    center = (w / 2, h / 2)
    camera_matrix = np.array([
        [focal_length, 0, center[0]],
        [0, focal_length, center[1]],
        [0, 0, 1]
    ])

    dist_coeffs = np.zeros((4, 1))

    _, rvec, _ = cv2.solvePnP(
        model_points, image_points, camera_matrix, dist_coeffs
    )

    rmat, _ = cv2.Rodrigues(rvec)
    angles, _, _, _, _, _ = cv2.RQDecomp3x3(rmat)

    pitch = angles[0] * 360
    yaw = angles[1] * 360
    roll = angles[2] * 360

    return pitch, yaw, roll


In [14]:
def extract_features(image_path):
    image = cv2.imread(image_path)
    if image is None:
        return None

    h, w, _ = image.shape
    rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    face_results = face_mesh.process(rgb)
    hand_results = hands.process(rgb)

    if not face_results.multi_face_landmarks:
        return None

    face_landmarks = face_results.multi_face_landmarks[0].landmark

    def get_point(idx):
        return np.array([
            face_landmarks[idx].x * w,
            face_landmarks[idx].y * h
        ])

    # =========================
    # EAR
    # =========================
    left_eye = [get_point(i) for i in LEFT_EYE]
    right_eye = [get_point(i) for i in RIGHT_EYE]

    ear_avg = (eye_aspect_ratio(left_eye) + eye_aspect_ratio(right_eye)) / 2

    # =========================
    # Eye State
    # =========================
    if ear_avg < 0.18:
        eye_state = 0   # CLOSED
    elif ear_avg < 0.23:
        eye_state = 1   # PARTIAL
    else:
        eye_state = 2   # OPEN

    # =========================
    # Gaze (SAFE – NO CRASH)
    # =========================
    try:
        left_iris = np.mean([get_point(i) for i in LEFT_IRIS], axis=0)
        right_iris = np.mean([get_point(i) for i in RIGHT_IRIS], axis=0)

        left_center = np.mean(left_eye, axis=0)
        right_center = np.mean(right_eye, axis=0)

        gaze_x = ((left_iris[0] - left_center[0]) +
                  (right_iris[0] - right_center[0])) / 2

        gaze_y = ((left_iris[1] - left_center[1]) +
                  (right_iris[1] - right_center[1])) / 2
    except:
        gaze_x, gaze_y = 0.0, 0.0

    # =========================
    # Head Pose (Pitch, Yaw, Roll)
    # =========================
    pitch, yaw, roll = get_head_pose(image, face_landmarks)

    # =========================
    # Hand–Face Distance
    # =========================
    hand_face_dist = 0.0
    if hand_results.multi_hand_landmarks:
        finger = hand_results.multi_hand_landmarks[0].landmark[8]
        finger_tip = np.array([
            finger.x * w,
            finger.y * h
        ])
        nose = get_point(NOSE_TIP)
        hand_face_dist = euclidean(finger_tip, nose)

    return [
        ear_avg,
        eye_state,
        gaze_x,
        gaze_y,
        pitch,
        yaw,
        roll,
        hand_face_dist
    ]


In [15]:
data_dir = "data"
output_csv = "features.csv"

with open(output_csv, "w", newline="") as f:
    writer = csv.writer(f)

    # CSV header
    writer.writerow([
        "EAR",
        "Eye_State",
        "Gaze_X",
        "Gaze_Y",
        "Head_Pitch",
        "Head_Yaw",
        "Head_Roll",
        "Hand_Face_Dist",
        "Label"
    ])

    # Loop through safe and unsafe folders
    for folder_name, label in [("safe", 0), ("unsafe", 1)]:
        folder_path = os.path.join(data_dir, folder_name)

        if not os.path.exists(folder_path):
            print(f"⚠️ Folder not found: {folder_path}")
            continue

        for img_name in os.listdir(folder_path):

            # ✅ FIX #3: skip non-image files
            if not img_name.lower().endswith((".jpg", ".jpeg", ".png")):
                continue

            img_path = os.path.join(folder_path, img_name)

            try:
                features = extract_features(img_path)
            except Exception as e:
                print(f"❌ Error processing {img_path}: {e}")
                continue

            if features is None:
                continue

            writer.writerow(features + [label])

print("✅ Feature extraction finished successfully")
print(f"📄 CSV saved as: {output_csv}")


✅ Feature extraction finished successfully
📄 CSV saved as: features.csv
